# 3. The Sleeper API in Python

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `02_configuration.ipynb` first if you haven't.

This notebook covers the [Sleeper API](https://docs.sleeper.com) itself -- what it is and what data it exposes -- and `SleeperClient`, the thin, paced, cached wrapper every other chapter in this tutorial is built on. Mirrors the [Sleeper API Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/03_sleeper_api.html) docs page.

## About the Sleeper API

Sleeper's API is public, free, read-only, and requires no signup, API key, or authentication of any kind -- every example below works as soon as you know a league ID. It is a plain JSON HTTP API at `https://api.sleeper.app`; the official reference lives at [docs.sleeper.com](https://docs.sleeper.com).

In exchange for free, unauthenticated access, Sleeper asks callers to be polite: stay under roughly 1000 requests per minute, and fetch the full player map (the largest endpoint, ~5 MB) no more than once a day. `SleeperClient` enforces both limits itself -- requests are paced automatically, and the player map is cached to disk with a 24-hour TTL -- so nothing in this tutorial needs to hand-roll throttling.

Data hangs off two roots: a **sport** (`nfl`) for the current state and the player map, and a **league** for everything else -- members, rosters, weekly matchups and transactions, and drafts. A draft has its own sub-resources (picks, traded picks) once one exists.

> **Why this matters:** unauthenticated doesn't mean unlimited, and it's easy to write a naive loop (fetch every week of every season individually, in a tight `for`) that looks fine on a small league and then gets you rate-limited on a real 6-season, 15-manager one. `SleeperClient`'s pacing exists specifically so you never have to notice this -- but understanding *why* it's there (a shared, free resource, politely used) matters if you ever write your own bulk-fetch loop outside this project.

## Connecting to a league

Use `SleeperClient` as a context manager so its HTTP session is closed for you. Every example below runs against the same real league used throughout this tutorial:

In [1]:
from nuclearff.sleeper import SleeperClient

league_id = "1367225133634191360"

with SleeperClient() as client:
    state = client.get_state("nfl")
    league = client.get_league(league_id)

state

{'week': 3,
 'season_type': 'regular',
 'season': '2026',
 'leg': 3,
 'league_season': '2026',
 'previous_season': '2025',
 'season_start_date': '2026-09-09',
 'display_week': 2,
 'league_create_season': '2026',
 'season_has_scores': True}

`state` is a small dict telling you which week's matchups or transactions to fetch next; `league` is the full settings object:

In [2]:
league["name"]

'NUCLEARFF REDRAFT'

In [3]:
league["roster_positions"]

['QB',
 'RB',
 'RB',
 'WR',
 'WR',
 'TE',
 'FLEX',
 'FLEX',
 'FLEX',
 'BN',
 'BN',
 'BN',
 'BN',
 'BN',
 'BN']

`roster_positions` lists one entry per roster slot -- this is exactly what `RosterSlots` (`02_configuration.ipynb`) counts to work out starter demand and replacement level.

### Who owns which roster

Rosters and users are separate endpoints, joined by `owner_id` / `user_id`:

In [4]:
with SleeperClient() as client:
    users = client.get_users(league_id)
    rosters = client.get_rosters(league_id)

names_by_user_id = {u["user_id"]: u["display_name"] for u in users}

for roster in sorted(rosters, key=lambda r: r["roster_id"]):
    owner = names_by_user_id.get(roster["owner_id"], "unknown")
    print(roster["roster_id"], owner)

1 casitzmann
2 hyoga10
3 nolmacdonald
4 ksavabi
5 aperry151
6 Donkeysride
7 ruhbberduhcky
8 jwhitney0220
9 nawfeastdallas
10 thatbolb


## The Sleeper NFL player map

`get_players()` returns every player Sleeper knows about -- roughly 5 MB of JSON -- keyed by Sleeper player ID:

In [5]:
with SleeperClient() as client:
    players = client.get_players()

players["4046"]

{'birth_state': None,
 'player_shard': '0',
 'birth_country': None,
 'oddsjam_id': 'B671C99CB711',
 'search_first_name': 'patrick',
 'search_last_name': 'mahomes',
 'team': 'KC',
 'sport': 'nfl',
 'number': 15,
 'rotowire_id': 11839,
 'injury_start_date': None,
 'active': True,
 'team_abbr': None,
 'sportradar_id': '11cad59d-90dd-449c-a839-dddaba4fe16c',
 'full_name': 'Patrick Mahomes',
 'player_id': '4046',
 'height': '74',
 'depth_chart_position': 'QB',
 'status': 'Active',
 'practice_description': None,
 'birth_date': '1995-09-17',
 'practice_participation': None,
 'last_name': 'Mahomes',
 'college': 'Texas Tech',
 'gsis_id': '00-0033873',
 'depth_chart_order': 1,
 'hashtag': '#patrickmahomes-NFL-KC-15',
 'competitions': [],
 'injury_body_part': None,
 'birth_city': None,
 'age': 31,
 'fantasy_data_id': 18890,
 'kalshi_id': '5f7253fb-e48e-44ff-9a51-4cd8e1893f68',
 'injury_notes': None,
 'fantasy_positions': ['QB'],
 'high_school': 'Whitehouse (TX)',
 'first_name': 'Patrick',
 'news_

Because this payload is large and Sleeper asks callers not to re-fetch it often, `get_players()` caches it to `<cache_dir>/sleeper_players_nfl.json` and only re-requests it once the cache is older than `players_ttl_hours` (24 hours by default). Pass `force_refresh=True` to bypass the cache.

Pass `position=` and/or `active=` to filter server-side and skip the full payload instead:

In [6]:
with SleeperClient() as client:
    active_qbs = client.get_players(position="QB", active=True)

len(active_qbs), len(players)

(355, 12228)

A filtered call always hits the network; the disk cache is specifically for the full unfiltered map.

## Trending players

`get_trending` returns the most-added or most-dropped players league-wide, right now -- but only ever a bare `player_id`. Resolve names yourself against the player map:

In [7]:
with SleeperClient() as client:
    trending = client.get_trending(kind="add", lookback_hours=24, limit=5)
    players = client.get_players()

for row in trending:
    p = players.get(row["player_id"], {})
    name = (
        p.get("full_name")
        or f"{p.get('first_name', '')} {p.get('last_name', '')}".strip()
        or row["player_id"]
    )
    print(f"{name:30s} count={row['count']}")

Emanuel Wilson                 count=1117242
Bryce Young                    count=618980
Tre Tucker                     count=558948
Tyler Shough                   count=534416
Adonai Mitchell                count=519096


Note the `first_name`/`last_name` fallback: a team defense has no `full_name` field in Sleeper's real payload, only the split fields -- if today's trending list includes one, it still resolves to a readable name above rather than crashing or printing a blank.

## Handling errors

A failed request raises a typed exception rather than a raw `requests` exception, so you can catch Sleeper-specific failures without also catching unrelated bugs: `SleeperHTTPError` for a non-retryable HTTP status (or once retries are exhausted on a 429 / 5xx), and `SleeperResponseError` if a response isn't the shape expected -- Sleeper returns a bare `null` for an unknown league or draft ID, for example, rather than a 404. Both subclass `SleeperAPIError` if you want to catch either:

In [8]:
from nuclearff.exceptions import SleeperAPIError

with SleeperClient() as client:
    try:
        client.get_league("not-a-real-league-id")
    except SleeperAPIError as exc:
        print(f"Sleeper call failed: {exc}")

Sleeper call failed: Sleeper request to https://api.sleeper.app/v1/league/not-a-real-league-id failed with 404 (Not Found)


> **Why this matters:** catching `SleeperAPIError` specifically -- instead of a bare `except Exception` -- matters the first time you write a script that loops over a list of league IDs from a file someone else typed by hand. A typo'd league ID should stop that one league and keep going; a real bug in your own code (a `KeyError` from assuming a field exists) should still crash loudly so you notice it. A blanket `except Exception` hides both the same way.

## What's Next

Everything above uses `SleeperClient` directly for one-off, interactive lookups. `04_capturing_a_league.ipynb` builds on the exact same client to capture a *complete*, reproducible league dataset -- a snapshot, full multi-season history, standings, matchups, transactions, and roster composition, all persisted so the rest of this tutorial can query them without hitting Sleeper again.

## See Also

- `02_configuration.ipynb` -- `LeagueConfig`, built from `get_league` above.
- [Sleeper API Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/03_sleeper_api.html) -- the matching docs page.